# MARV × Titans — find it, edit it, measure the damage (Colab, no GPU needed)

MARV's whole point on a frozen model: find where a fact lives, edit it, measure what broke.
`titans_per_unit.py` showed that giving a Titans-style memory's units their own individual
forget rates makes real per-unit localization appear — a slow-forgetting unit reliably becomes
the dedicated home for one specific fact (confirmed across 8 seeds).

This notebook does the actual editing move on that same setup: solve for a new output row on
the unit that owns a fact, so it now recalls something completely different — then check what
else moved. No training, no GPU — this runs in milliseconds, the same as `titans_per_unit.py` itself.

In [ ]:
!git clone -q -b marv-titan https://github.com/thebnbrkr/marv.git /content/marv
import sys; sys.path.insert(0, '/content/marv/experiments')

import torch, numpy as np
from titans_per_unit import (
    HIDDEN, DIM, store_tracked_pairs, run_ablation_sweep, recall_cosines,
    edit_unit_to_target, run_editing_demo, report,
)
print('ready')

## 1. Store some facts, find a unit that owns one of them

Same setup as `titans_per_unit.py`: 12 tracked key/value pairs, each unit assigned its own
decay rate spread across [0, 1] (the thing the real Titans architecture doesn't allow, since its
forget gate is one shared number for all units — see `experiments/README.md` roadmap item 1c).

In [ ]:
n_pairs = 12
seed = 0
g = torch.Generator().manual_seed(seed)
pairs = torch.randn(n_pairs, DIM, generator=g)
spread_decay = torch.linspace(0.02, 0.98, HIDDEN)

w0, w1 = store_tracked_pairs(pairs, spread_decay, seed)
baseline, drop = run_ablation_sweep(w0, w1, pairs)
report('SPREAD decay', spread_decay, baseline, drop)

**Look at the printed "most causally important units"** — pick one that breaks exactly one pair hard (shown in the `breaks pairs [...]` column). The default below uses unit 4 / pair 0, which owns pair 0 in this exact seed — if you change the seed above, check the printout and update the numbers in the next cell to match what you actually find.

## 2. Edit it — retarget that unit's output, not just delete it

`edit_unit_to_target` solves for a new output row on the chosen unit so the edited pair now recalls a brand-new target — here, we make pair 0 recall pair 5's value instead, using only unit 4's own weights.

In [ ]:
run_editing_demo(pairs, spread_decay, seed=seed, unit=4, edit_pair_idx=0, target=pairs[5])

**How to read it:** the edited pair's recall of its new target should be ~1.000 — the edit is always mathematically exact, since it's solved for directly. The real thing to look at is the `<-- collateral` pairs: how many *other* facts moved, and by how much, even though we only touched one unit.

## 3. Does picking a MORE isolated unit reduce the damage?

Unit 4 was flagged as owning pair 0, but ablation only tells you it's the *biggest* contributor — not the *only* one. This searches all 256 units × 12 pairs for whichever (unit, pair) pair looks most cleanly isolated by ablation (highest ratio of effect-on-its-pair to effect-on-everything-else), then edits that one instead, to see if being more "specific" by that measure actually buys less collateral damage.

In [ ]:
best = []
for sd in range(8):
    gg = torch.Generator().manual_seed(sd)
    pp = torch.randn(n_pairs, DIM, generator=gg)
    ww0, ww1 = store_tracked_pairs(pp, spread_decay, sd)
    bb, dd = run_ablation_sweep(ww0, ww1, pp)
    absdrop = np.abs(dd)
    for u in range(HIDDEN):
        tp = absdrop[u].argmax()
        te = absdrop[u, tp]
        oe = absdrop[u].sum() - te
        if te < 0.1:
            continue
        best.append((te / (oe + 1e-9), te, sd, u, tp))

best.sort(reverse=True)
print('most ablation-specific (unit, pair) found:', best[0])
spec, te, best_seed, best_unit, best_pair = best[0]

gb = torch.Generator().manual_seed(best_seed)
pairs_b = torch.randn(n_pairs, DIM, generator=gb)
swap_target = pairs_b[(best_pair + 4) % n_pairs]  # some other pair's value
run_editing_demo(pairs_b, spread_decay, seed=best_seed, unit=best_unit, edit_pair_idx=best_pair, target=swap_target)

## What this shows

The edit itself always works exactly — that part is just linear algebra. The finding is collateral: editing one "owning" unit disturbs several other stored facts, and picking a MORE ablation-specific unit does not reliably reduce that (in the local run behind this notebook's defaults: specificity 1.75 → 0.473 max collateral; specificity 2.95 → 0.540, worse). Ablation-specificity measures how small a unit's *existing* contribution to other pairs is; editing-collateral measures how much that unit's *activation* fires for other pairs' own keys, independent of how small its old content there was — different properties of the same unit, and one doesn't predict the other.

This is the same collateral-damage story MARV's own docs already tell about frozen models — "one neuron is shared by many unrelated facts" — now shown causally on a *live*, self-updating memory instead. See `experiments/README.md` roadmap item 1d on the `marv-titan` branch.